# CWR 2024–2026 encounters: ecotype report and CSV export

Query the public Atlist sources behind the Center for Whale Research encounter maps for [2024](https://www.whaleresearch.com/encounters2024), [2025](https://www.whaleresearch.com/encounters), and [2026](https://www.whaleresearch.com/encounters-map-2026). The notebook normalizes encounter markers, maps them by ecotype, plots monthly encounter counts, exports one combined CSV, and builds an interactive HTML report.

**Coordinate interpretation:** Atlist provides one signed decimal latitude/longitude pair per marker. The source does not label it as an encounter start or end, so the export uses `map_lat`/`map_lon` and `coordinate_role = map_marker_unspecified`.

**Counting interpretation:** each row is one CWR/Atlist encounter marker, not an independent animal sighting and not an effort-corrected observation.

In [ ]:
from __future__ import annotations

import hashlib
import html
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import folium
import pandas as pd
import plotly.express as px
import requests
from IPython.display import HTML, display


In [ ]:
YEAR_SOURCES = {
    2024: {
        "page_url": "https://www.whaleresearch.com/encounters2024",
        "map_id": "3d6d0c96-06fb-4087-959c-1ecdcab167af",
    },
    2025: {
        "page_url": "https://www.whaleresearch.com/encounters",
        "map_id": "6d677a80-8764-481c-9228-abcb73cc56eb",
    },
    2026: {
        "page_url": "https://www.whaleresearch.com/encounters-map-2026",
        "map_id": "4fdb0211-7443-48fb-9334-2c401932b4ba",
    },
}
REQUEST_TIMEOUT_SECONDS = 30

import os
NOTEBOOK_DIR = Path(os.environ["MARINE_MAMMALS_CWR_OUTPUT_ROOT"]).expanduser().resolve()
EXPORT_DIR = NOTEBOOK_DIR / "exports"
REPORT_DIR = NOTEBOOK_DIR / "reports"
CSV_PATH = EXPORT_DIR / "cwr_2024_2026_atlist_encounters.csv"
REPORT_PATH = REPORT_DIR / "cwr_2024_2026_encounter_report.html"

session = requests.Session()
session.headers.update({"User-Agent": "OrcaCast-CWR-map-export/0.2 (bounded research request)"})
print({
    "source_years": list(YEAR_SOURCES),
    "csv_path": str(CSV_PATH.resolve()),
    "report_path": str(REPORT_PATH.resolve()),
})


## Query the public Atlist sources

Each source is fetched with direct public GET requests and no credentials, cookies, request body, or browser state. The marker-response checksum and map update time are carried into every exported row for provenance.

In [ ]:
retrieved_at_utc = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

source_payloads: dict[int, dict[str, Any]] = {}
request_evidence: list[dict[str, Any]] = []

for source_year, source in YEAR_SOURCES.items():
    map_id = source["map_id"]
    api_root = f"https://api.atlist.com/v1/map/{map_id}"
    metadata_url = f"{api_root}/fields"
    markers_url = f"{api_root}/markers"

    map_response = session.get(metadata_url, timeout=REQUEST_TIMEOUT_SECONDS)
    map_response.raise_for_status()
    map_metadata = map_response.json()

    markers_response = session.get(markers_url, timeout=REQUEST_TIMEOUT_SECONDS)
    markers_response.raise_for_status()
    markers_payload_sha256 = hashlib.sha256(markers_response.content).hexdigest()
    markers = markers_response.json().get("markers", [])

    source_payloads[source_year] = {
        **source,
        "map_url": f"https://my.atlist.com/map/{map_id}?share=true",
        "metadata_url": metadata_url,
        "markers_url": markers_url,
        "map_metadata": map_metadata,
        "markers": markers,
        "markers_payload_sha256": markers_payload_sha256,
    }
    request_evidence.append({
        "source_year": source_year,
        "status": markers_response.status_code,
        "content_type": markers_response.headers.get("content-type"),
        "authorization_sent": "Authorization" in markers_response.request.headers,
        "cookie_sent": "Cookie" in markers_response.request.headers,
        "request_body_sent": markers_response.request.body is not None,
        "map_name": map_metadata.get("name"),
        "map_updated_at": map_metadata.get("updatedAt"),
        "retrieved_at_utc": retrieved_at_utc,
        "raw_marker_count": len(markers),
        "markers_payload_sha256": markers_payload_sha256,
    })

request_evidence_df = pd.DataFrame(request_evidence)
display(request_evidence_df)


## Normalize encounter markers

The parser retains source text and applies a few explicit year-specific harmonizations:

- 2024 tag labels are mapped to the same canonical ecotype names used in 2025–2026.
- `Encounter summary` and `EncSummary` are treated as the same source field.
- Missing, two-digit, or malformed title years are parsed with an explicit QC flag; the unmodified title date remains in `date_source_text`.
- Non-encounter visitor-center pins are excluded and recorded separately.

Unavailable values remain null. Times are normalized to 24-hour clock where parseable, but the time zone remains null because it is not stated by the source.

In [ ]:
ENCOUNTER_NAME_PATTERN = re.compile(r"^Encounter\s+#(?P<number>[^-]+?)\s+-\s+(?P<date>.+?)\s*$", re.I)
NOTE_LABEL_PATTERN = re.compile(
    r"(?im)^[\s\u200b]*(EncSummary|Encounter\s*Summary|ObservBegin|ObservEnd|Vessel|Staff|Other\s+Observers|Pods|IDs\s*Encountered|LocationDescr)\s*:\s*"
)
NOTE_LABEL_CANONICAL = {
    "encsummary": "EncSummary",
    "encountersummary": "EncSummary",
    "observbegin": "ObservBegin",
    "observend": "ObservEnd",
    "vessel": "Vessel",
    "staff": "Staff",
    "otherobservers": "Other Observers",
    "pods": "Pods",
    "idsencountered": "IDsEncountered",
    "locationdescr": "LocationDescr",
}
ECOTYPE_ALIASES = {
    "Southern Resident": "Southern Resident Killer Whales",
    "Southern Resident Killer Whales": "Southern Resident Killer Whales",
    "Northern Resident Killer Whales": "Northern Resident Killer Whales",
    "Bigg's killer whales": "Bigg's Killer Whales",
    "Bigg's Killer Whales": "Bigg's Killer Whales",
}
ECOTYPE_ORDER = [
    "Southern Resident Killer Whales",
    "Bigg's Killer Whales",
    "Northern Resident Killer Whales",
    "Unknown / not stated",
]


def html_to_lines(value: str | None) -> str:
    if not value:
        return ""
    text = re.sub(r"(?i)<br\s*/?>", "\n", value)
    text = re.sub(r"(?i)</p\s*>", "\n", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = html.unescape(text).replace("\xa0", " ")
    return "\n".join(line.strip() for line in text.splitlines() if line.strip())


def parse_labeled_notes(notes_html: str | None) -> dict[str, str]:
    text = html_to_lines(notes_html)
    matches = list(NOTE_LABEL_PATTERN.finditer(text))
    fields: dict[str, str] = {}
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        normalized_label = re.sub(r"\s+", "", match.group(1)).casefold()
        fields[NOTE_LABEL_CANONICAL[normalized_label]] = text[match.end():end].strip()
    return fields


def split_values(value: str | None) -> list[str]:
    if not value:
        return []
    return [
        re.sub(r"(?i)^and\s+", "", item.strip())
        for item in re.split(r"[,;]\s*", value)
        if item.strip()
    ]


def normalize_time(value: str | None) -> str | None:
    if not value:
        return None
    compact = re.sub(r"\s+", " ", value.strip()).upper()
    for fmt in ("%I:%M %p", "%I %p", "%H:%M", "%H%M"):
        try:
            return datetime.strptime(compact, fmt).strftime("%H:%M")
        except ValueError:
            pass
    return None


def parse_encounter_date(date_source_text: str | None, source_year: int) -> tuple[str | None, str | None, list[str]]:
    if not date_source_text:
        return None, None, ["missing_encounter_date_text"]

    parsed_text = date_source_text.strip()
    qc_flags: list[str] = []
    parse_steps: list[str] = []

    # Common source spelling; this is semantically equivalent rather than a repair.
    parsed_text = re.sub(r"\bSept\b", "Sep", parsed_text, flags=re.I)

    malformed_punctuation = (
        ",," in parsed_text
        or bool(re.search(r",(?=\d{4}$)", parsed_text))
        or bool(re.match(r"^[A-Za-z]{3}\.", parsed_text))
    )
    parsed_text = parsed_text.replace(",,", ",")
    parsed_text = re.sub(r",(?=\d{4}$)", ", ", parsed_text)
    parsed_text = re.sub(r"^([A-Za-z]{3})\.", r"\1", parsed_text)
    if malformed_punctuation:
        qc_flags.append("date_text_normalized_for_parse")
        parse_steps.append("punctuation_normalized")

    if re.search(r",\s*\d{3}$", parsed_text):
        parsed_text = re.sub(r",\s*\d{3}$", f", {source_year}", parsed_text)
        qc_flags.append("date_year_repaired_from_source_map")
        parse_steps.append("map_year_repaired")
    elif re.search(r",\s*\d{2}$", parsed_text):
        qc_flags.append("date_two_digit_year_expanded")
        parse_steps.append("two_digit_year_expanded")
    elif not re.search(r"\b\d{4}\b", parsed_text):
        parsed_text = f"{parsed_text}, {source_year}"
        qc_flags.append("date_year_inferred_from_source_map")
        parse_steps.append("map_year_inferred")

    encounter_date = None
    for fmt in ("%b %d, %Y", "%B %d, %Y", "%b %d, %y", "%B %d, %y"):
        try:
            encounter_date = datetime.strptime(parsed_text, fmt).date().isoformat()
            break
        except ValueError:
            pass
    if encounter_date is None:
        qc_flags.append("unparsed_encounter_date")

    parse_method = "source_title" + ("_with_" + "_and_".join(parse_steps) if parse_steps else "")
    return encounter_date, parse_method, qc_flags


def marker_tags(marker: dict[str, Any]) -> list[str]:
    values: list[str] = []
    for tag in marker.get("tags") or []:
        if isinstance(tag, str):
            values.append(tag)
        elif isinstance(tag, dict) and tag.get("name"):
            values.append(str(tag["name"]))
    return values


def normalize_marker(marker: dict[str, Any], source_year: int, source: dict[str, Any]) -> dict[str, Any] | None:
    name = marker.get("name") or ""
    name_match = ENCOUNTER_NAME_PATTERN.match(name)
    if not name_match:
        return None

    encounter_number = name_match.group("number").strip()
    date_source_text = name_match.group("date").strip()
    encounter_date, date_parse_method, date_qc_flags = parse_encounter_date(date_source_text, source_year)

    note_fields = parse_labeled_notes(marker.get("notes"))
    tags = marker_tags(marker)
    ecotype = next((ECOTYPE_ALIASES[tag] for tag in tags if tag in ECOTYPE_ALIASES), None)
    pods = split_values(note_fields.get("Pods"))
    if not pods:
        pods = [tag.removesuffix(" Pod") for tag in tags if tag.endswith(" Pod")]

    map_lat = float(marker["lat"]) if marker.get("lat") is not None else None
    map_lon = float(marker["long"]) if marker.get("long") is not None else None
    qc_flags = list(date_qc_flags)
    if encounter_date and int(encounter_date[:4]) != source_year:
        qc_flags.append("map_year_date_mismatch")
    if map_lat is None or map_lon is None:
        qc_flags.append("missing_map_coordinate")
    elif not (-90 <= map_lat <= 90 and -180 <= map_lon <= 180):
        qc_flags.append("invalid_map_coordinate")

    map_metadata = source["map_metadata"]
    source_record_id = marker.get("id")
    return {
        "source": "center_for_whale_research",
        "source_system": "atlist",
        "source_year": source_year,
        "source_map_id": source["map_id"],
        "source_record_id": source_record_id,
        "source_record_key": f"{source['map_id']}:{source_record_id}",
        "source_record_name": name,
        "record_type": "encounter_marker",
        "encounter_id": None,
        "encounter_number": encounter_number,
        "date": encounter_date,
        "date_source_text": date_source_text,
        "date_parse_method": date_parse_method,
        "start_time": normalize_time(note_fields.get("ObservBegin")),
        "start_time_source_text": note_fields.get("ObservBegin") or None,
        "end_time": normalize_time(note_fields.get("ObservEnd")),
        "end_time_source_text": note_fields.get("ObservEnd") or None,
        "time_zone": None,
        "ecotype": ecotype,
        "pods": pods,
        "individuals": split_values(note_fields.get("IDsEncountered")),
        "location_description": note_fields.get("LocationDescr") or None,
        "map_lat": map_lat,
        "map_lon": map_lon,
        "coordinate_crs": "EPSG:4326",
        "coordinate_role": "map_marker_unspecified" if map_lat is not None and map_lon is not None else None,
        "start_lat": None,
        "start_lon": None,
        "end_lat": None,
        "end_lon": None,
        "summary": note_fields.get("EncSummary") or None,
        "source_tags": tags,
        "source_url": source["page_url"],
        "source_map_url": source["map_url"],
        "source_endpoint_url": source["markers_url"],
        "source_record_created_at": marker.get("createdAt"),
        "source_record_updated_at": marker.get("updatedAt"),
        "source_map_created_at": map_metadata.get("createdAt"),
        "source_map_updated_at": map_metadata.get("updatedAt"),
        "retrieved_at_utc": retrieved_at_utc,
        "source_payload_sha256": source["markers_payload_sha256"],
        "qc_flags": qc_flags,
    }


In [ ]:
records: list[dict[str, Any]] = []
excluded_markers: list[dict[str, Any]] = []

for source_year, source in source_payloads.items():
    for marker in source["markers"]:
        record = normalize_marker(marker, source_year, source)
        if record is None:
            excluded_markers.append({
                "source_year": source_year,
                "source_map_id": source["map_id"],
                "source_record_id": marker.get("id"),
                "source_record_name": marker.get("name"),
                "exclusion_reason": "non_encounter_marker_title",
            })
        else:
            records.append(record)

records.sort(key=lambda row: (row["date"] or "9999-99-99", row["source_year"], row["encounter_number"] or ""))
encounters = pd.DataFrame(records)
excluded_markers_df = pd.DataFrame(excluded_markers)

assert encounters["source_record_id"].notna().all()
assert encounters["source_record_key"].is_unique
assert encounters["map_lat"].dropna().between(-90, 90).all()
assert encounters["map_lon"].dropna().between(-180, 180).all()
assert encounters["date"].notna().all(), "All encounter titles should have a parseable date"
assert set(encounters["source_year"]) == set(YEAR_SOURCES)

display(encounters[[
    "source_year", "encounter_number", "date", "date_source_text", "ecotype",
    "pods", "location_description", "map_lat", "map_lon", "qc_flags",
]].head(10))
print({"encounter_records": len(encounters), "excluded_non_encounter_markers": len(excluded_markers_df)})
display(excluded_markers_df)


## Summary metrics and monthly time series

The monthly series uses the source title date. Counts are zero-filled only from January 2024 through the latest in-year 2026 encounter month returned by the API at pull time; later 2026 months are not shown as zero. The one source-year/date mismatch remains on its source-stated date and is flagged in the export.

In [ ]:
encounters["ecotype_display"] = encounters["ecotype"].fillna("Unknown / not stated")
encounters["date_dt"] = pd.to_datetime(encounters["date"], errors="coerce")
encounters["month"] = encounters["date_dt"].dt.to_period("M").dt.to_timestamp()

ecotype_counts = (
    encounters.groupby("ecotype_display", as_index=False, observed=True)
    .agg(encounters=("source_record_key", "count"))
    .sort_values("encounters", ascending=False)
)
year_ecotype_counts = (
    encounters.pivot_table(
        index="source_year",
        columns="ecotype_display",
        values="source_record_key",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(columns=[column for column in ECOTYPE_ORDER if column in encounters["ecotype_display"].unique()], fill_value=0)
    .astype(int)
)
year_ecotype_counts["Total"] = year_ecotype_counts.sum(axis=1)
year_ecotype_counts.columns.name = None

latest_in_year_date = encounters.loc[
    encounters["date_dt"].dt.year.eq(encounters["source_year"]), "date_dt"
].max()
series_months = pd.date_range("2024-01-01", latest_in_year_date.to_period("M").to_timestamp(), freq="MS")
series_ecotypes = [value for value in ECOTYPE_ORDER if value in encounters["ecotype_display"].unique()]
series_grid = pd.MultiIndex.from_product(
    [series_months, series_ecotypes], names=["month", "ecotype_display"]
).to_frame(index=False)
monthly_observed = (
    encounters.groupby(["month", "ecotype_display"], as_index=False, observed=True)
    .agg(encounters=("source_record_key", "count"))
)
monthly_counts = series_grid.merge(monthly_observed, how="left", on=["month", "ecotype_display"])
monthly_counts["encounters"] = monthly_counts["encounters"].fillna(0).astype(int)

display(ecotype_counts)
display(year_ecotype_counts)

ECOTYPE_COLORS = {
    "Southern Resident Killer Whales": "#0072B2",
    "Bigg's Killer Whales": "#D55E00",
    "Northern Resident Killer Whales": "#009E73",
    "Unknown / not stated": "#6B7280",
}
time_series_fig = px.line(
    monthly_counts,
    x="month",
    y="encounters",
    color="ecotype_display",
    color_discrete_map=ECOTYPE_COLORS,
    category_orders={"ecotype_display": ECOTYPE_ORDER},
    markers=True,
    labels={"month": "Encounter month", "encounters": "Encounter markers", "ecotype_display": "Ecotype"},
    title="Monthly CWR encounter markers by ecotype",
)
time_series_fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    legend_title_text="Ecotype",
    margin=dict(l=50, r=30, t=75, b=50),
)
time_series_fig.update_yaxes(rangemode="tozero", dtick=5)
time_series_fig.show()


## Encounter map by ecotype

Marker color represents ecotype, while layer controls toggle source years. Popups show source year, encounter number, title date, ecotype, pods, time window, location, and the Atlist record ID without reproducing the full encounter narrative.

In [ ]:
mapped = encounters.dropna(subset=["map_lat", "map_lon"]).copy()
center = [mapped["map_lat"].mean(), mapped["map_lon"].mean()]

encounter_map = folium.Map(
    location=center,
    zoom_start=7,
    tiles="CartoDB positron",
    control_scale=True,
)

for source_year, group in mapped.groupby("source_year", sort=True):
    layer = folium.FeatureGroup(name=f"{source_year} ({len(group)})", show=True)
    for row in group.itertuples(index=False):
        ecotype = row.ecotype_display
        color = ECOTYPE_COLORS.get(ecotype, ECOTYPE_COLORS["Unknown / not stated"])
        pods = ", ".join(row.pods) if row.pods else "Not stated"
        time_window = " – ".join(
            str(value) for value in (row.start_time, row.end_time)
            if pd.notna(value) and str(value).strip()
        ) or "Not stated"
        location_description = row.location_description if pd.notna(row.location_description) else "Not stated"
        popup_html = (
            f"<strong>{html.escape(str(row.source_year))} encounter #{html.escape(str(row.encounter_number or 'unknown'))}</strong><br>"
            f"Date: {html.escape(str(row.date or 'Not stated'))}<br>"
            f"Ecotype: {html.escape(ecotype)}<br>"
            f"Pods: {html.escape(pods)}<br>"
            f"Time: {html.escape(time_window)} (time zone not stated)<br>"
            f"Location: {html.escape(str(location_description))}<br>"
            f"Atlist record: {html.escape(str(row.source_record_id))}"
        )
        folium.CircleMarker(
            location=[row.map_lat, row.map_lon],
            radius=5.2,
            color="#FFFFFF",
            weight=1.1,
            fill=True,
            fill_color=color,
            fill_opacity=0.84,
            tooltip=f"{row.source_year} encounter #{row.encounter_number} · {row.date}",
            popup=folium.Popup(popup_html, max_width=400),
        ).add_to(layer)
    layer.add_to(encounter_map)

folium.LayerControl(collapsed=False).add_to(encounter_map)
legend_items = "".join(
    f'<div style="margin:4px 0"><span style="display:inline-block;width:11px;height:11px;border-radius:50%;background:{color};margin-right:7px"></span>{html.escape(label)}</div>'
    for label, color in ECOTYPE_COLORS.items()
    if label in mapped["ecotype_display"].unique()
)
legend = f'''
<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;padding:10px 12px;border:1px solid #CBD5E1;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.15);font:12px/1.3 sans-serif">
<strong>Ecotype</strong>{legend_items}
</div>
'''
encounter_map.get_root().html.add_child(folium.Element(legend))
encounter_map


## Combined CSV export

The CSV contains one encounter marker per row. Arrays are serialized as compact JSON strings so they reload unambiguously. Blank values mean unavailable or unstated—not zero.

In [ ]:
EXPORT_COLUMNS = [
    "source", "source_system", "source_year", "source_map_id", "source_record_id",
    "source_record_key", "source_record_name", "record_type",
    "encounter_id", "encounter_number", "date", "date_source_text", "date_parse_method",
    "start_time", "start_time_source_text", "end_time", "end_time_source_text", "time_zone",
    "ecotype", "pods", "individuals", "location_description",
    "map_lat", "map_lon", "coordinate_crs", "coordinate_role",
    "start_lat", "start_lon", "end_lat", "end_lon",
    "summary", "source_tags", "source_url", "source_map_url", "source_endpoint_url",
    "source_record_created_at", "source_record_updated_at",
    "source_map_created_at", "source_map_updated_at", "retrieved_at_utc",
    "source_payload_sha256", "qc_flags",
]
JSON_COLUMNS = ["pods", "individuals", "source_tags", "qc_flags"]

export_df = encounters[EXPORT_COLUMNS].copy()
for column in JSON_COLUMNS:
    export_df[column] = export_df[column].map(
        lambda value: json.dumps(value, ensure_ascii=False, separators=(",", ":"))
    )

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_df.to_csv(CSV_PATH, index=False, lineterminator="\n")
csv_sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()

reloaded = pd.read_csv(
    CSV_PATH,
    dtype={"encounter_number": "string", "source_record_id": "string", "source_record_key": "string"},
)
assert len(reloaded) == len(export_df)
assert reloaded["source_record_key"].nunique(dropna=True) == len(export_df)
assert pd.to_numeric(reloaded["map_lat"], errors="coerce").between(-90, 90).all()
assert pd.to_numeric(reloaded["map_lon"], errors="coerce").between(-180, 180).all()
assert set(JSON_COLUMNS).issubset(reloaded.columns)

print({
    "csv_path": str(CSV_PATH.resolve()),
    "rows": len(reloaded),
    "columns": len(reloaded.columns),
    "csv_bytes": CSV_PATH.stat().st_size,
    "csv_sha256": csv_sha256,
})
display(reloaded[[
    "source_year", "source_record_id", "encounter_number", "date", "ecotype",
    "map_lat", "map_lon", "qc_flags",
]].head())


## HTML report

The report is a single HTML file containing metrics, source-year and ecotype tables, the monthly time series, the interactive map, QC details, pull metadata, and a link to the combined CSV. Plotly is embedded; the Folium map loads its basemap and Leaflet assets from their normal internet sources.

In [ ]:
def dataframe_html(frame: pd.DataFrame, index: bool = False) -> str:
    return frame.to_html(index=index, border=0, classes="data-table", escape=True)


qc_rows = encounters.loc[
    encounters["qc_flags"].map(bool),
    ["source_year", "encounter_number", "source_record_name", "date", "qc_flags"],
].copy()
qc_rows["qc_flags"] = qc_rows["qc_flags"].map(lambda values: ", ".join(values))

coverage_rows = []
for source_year, source in source_payloads.items():
    year_rows = encounters.loc[encounters["source_year"].eq(source_year)].copy()
    in_year_dates = year_rows.loc[year_rows["date_dt"].dt.year.eq(source_year), "date_dt"]
    coverage_rows.append({
        "Source year": source_year,
        "Raw Atlist markers": len(source["markers"]),
        "Encounter markers": len(year_rows),
        "Excluded non-encounter pins": int(excluded_markers_df["source_year"].eq(source_year).sum()),
        "First in-year encounter": in_year_dates.min().date().isoformat() if len(in_year_dates) else None,
        "Last in-year encounter": in_year_dates.max().date().isoformat() if len(in_year_dates) else None,
        "QC-flagged encounters": int(year_rows["qc_flags"].map(bool).sum()),
    })
coverage_df = pd.DataFrame(coverage_rows)

year_report = year_ecotype_counts.reset_index().rename(columns={"source_year": "Source year"})
ecotype_report = ecotype_counts.rename(columns={"ecotype_display": "Ecotype", "encounters": "Encounter markers"})

time_series_html = time_series_fig.to_html(
    full_html=False,
    include_plotlyjs=True,
    config={"displaylogo": False, "responsive": True},
)
map_document = encounter_map.get_root().render()
map_srcdoc = html.escape(map_document, quote=True)

source_links = "".join(
    f'<li><a href="{html.escape(source["page_url"], quote=True)}">CWR {year} encounter page</a> · '
    f'<a href="{html.escape(source["map_url"], quote=True)}">Atlist map</a></li>'
    for year, source in source_payloads.items()
)
metric_cards = "".join(
    f'<article class="metric"><span>{html.escape(str(label))}</span><strong>{html.escape(str(value))}</strong></article>'
    for label, value in [
        ("Encounter markers", f"{len(encounters):,}"),
        ("Source years", "2024–2026"),
        ("Ecotypes", encounters["ecotype_display"].nunique()),
        ("QC-flagged", int(encounters["qc_flags"].map(bool).sum())),
        ("Pull date (UTC)", retrieved_at_utc),
    ]
)

report_html = f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>CWR 2024–2026 encounter report</title>
<style>
:root {{ --ink:#102A43; --muted:#52667A; --line:#D9E2EC; --panel:#F6F9FC; --accent:#0072B2; }}
* {{ box-sizing:border-box; }}
body {{ margin:0; color:var(--ink); background:#EDF3F8; font:15px/1.55 Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }}
main {{ width:min(1180px, calc(100% - 32px)); margin:32px auto 64px; }}
header, section {{ background:white; border:1px solid var(--line); border-radius:14px; box-shadow:0 8px 24px rgba(16,42,67,.06); }}
header {{ padding:32px; border-top:5px solid var(--accent); }}
section {{ margin-top:20px; padding:26px; }}
h1 {{ margin:0 0 8px; font-size:clamp(26px,4vw,42px); line-height:1.12; letter-spacing:-.02em; }}
h2 {{ margin:0 0 14px; font-size:23px; }}
h3 {{ margin:24px 0 10px; font-size:17px; }}
p {{ max-width:84ch; }}
.lede {{ color:var(--muted); font-size:17px; margin:0; }}
.metrics {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:12px; margin-top:24px; }}
.metric {{ padding:15px 16px; background:var(--panel); border:1px solid var(--line); border-radius:10px; }}
.metric span {{ display:block; color:var(--muted); font-size:12px; font-weight:700; letter-spacing:.04em; text-transform:uppercase; }}
.metric strong {{ display:block; margin-top:5px; font-size:20px; overflow-wrap:anywhere; }}
.two-col {{ display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:20px; align-items:start; }}
.scroll {{ overflow-x:auto; }}
.data-table {{ width:100%; border-collapse:collapse; font-size:14px; }}
.data-table th, .data-table td {{ padding:9px 10px; text-align:left; border-bottom:1px solid var(--line); vertical-align:top; }}
.data-table th {{ background:var(--panel); font-weight:700; white-space:nowrap; }}
.chart {{ min-height:480px; }}
.map-frame {{ width:100%; min-height:650px; border:1px solid var(--line); border-radius:10px; background:#E9F1F7; }}
.callout {{ padding:14px 16px; border-left:4px solid var(--accent); background:#EDF7FC; border-radius:6px; color:#234E67; }}
a {{ color:#005A8D; }}
code {{ background:#EEF2F6; padding:.1em .3em; border-radius:4px; }}
footer {{ color:var(--muted); margin-top:20px; font-size:13px; text-align:center; }}
@media (max-width:800px) {{ .two-col {{ grid-template-columns:1fr; }} header, section {{ padding:20px; }} .map-frame {{ min-height:520px; }} }}
</style>
</head>
<body>
<main>
<header>
  <h1>CWR 2024–2026 encounter report</h1>
  <p class="lede">Public Center for Whale Research Atlist encounter markers summarized by source year and ecotype.</p>
  <div class="metrics">{metric_cards}</div>
</header>

<section>
  <h2>At a glance</h2>
  <div class="two-col">
    <div class="scroll"><h3>Total by ecotype</h3>{dataframe_html(ecotype_report)}</div>
    <div class="scroll"><h3>Source-year totals</h3>{dataframe_html(year_report)}</div>
  </div>
  <p class="callout">These are encounter-marker counts, not independent whale sightings, abundance estimates, or effort-corrected occurrence rates. The 2026 map is partial at the pull date.</p>
</section>

<section>
  <h2>Monthly encounter markers</h2>
  <p>Counts use encounter dates parsed from source titles. Zeroes are filled only from January 2024 through {latest_in_year_date.strftime('%B %Y')}, the latest in-year 2026 encounter month returned at this pull. Later months are not represented as zero.</p>
  <div class="chart">{time_series_html}</div>
</section>

<section>
  <h2>Encounter map</h2>
  <p>Color identifies ecotype; the layer control toggles source years. Coordinates are Atlist marker locations with an unspecified role.</p>
  <iframe class="map-frame" title="CWR encounter map by ecotype and source year" srcdoc="{map_srcdoc}"></iframe>
</section>

<section>
  <h2>Coverage and quality controls</h2>
  <div class="scroll">{dataframe_html(coverage_df)}</div>
  <h3>QC-flagged encounter titles</h3>
  <div class="scroll">{dataframe_html(qc_rows) if len(qc_rows) else '<p>None.</p>'}</div>
  <h3>Excluded non-encounter pins</h3>
  <div class="scroll">{dataframe_html(excluded_markers_df) if len(excluded_markers_df) else '<p>None.</p>'}</div>
</section>

<section>
  <h2>Data and provenance</h2>
  <p><a href="../exports/{CSV_PATH.name}">Download the combined CSV</a> ({len(export_df):,} rows; SHA-256 <code>{csv_sha256}</code>).</p>
  <p>Pull time: <strong>{html.escape(retrieved_at_utc)}</strong>. Each CSV row retains its CWR page, Atlist map and API endpoint URLs, source map and record timestamps, response checksum, original date text, parse method, and QC flags.</p>
  <ul>{source_links}</ul>
  <p>Blank fields mean the source did not state a value. Time zones and start/end coordinate roles are not inferred. Confirm permission and redistribution terms before publishing source narratives or individual-whale details.</p>
</section>
<footer>Generated by <code>{html.escape(str(Path(__file__).name if '__file__' in globals() else '02_CWR_2024_2026_ECOTYPE_REPORT_AND_EXPORT.ipynb'))}</code></footer>
</main>
</body>
</html>
'''

REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(report_html, encoding="utf-8")
report_sha256 = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()

print({
    "report_path": str(REPORT_PATH.resolve()),
    "report_bytes": REPORT_PATH.stat().st_size,
    "report_sha256": report_sha256,
    "map_markers": len(mapped),
    "time_series_months": len(series_months),
    "qc_flagged_records": len(qc_rows),
})
display(HTML(f'<a href="{REPORT_PATH.resolve().as_uri()}" target="_blank">Open generated HTML report</a>'))


## Interpretation notes

- The map shows CWR/Atlist encounter marker locations, not independent whale sightings within an encounter.
- Marker counts are reporting records and should not be interpreted as effort-corrected whale abundance or occurrence rates.
- `encounter_id` and start/end coordinates remain null until a stable cross-year CWR identity and coordinate semantics are established.
- Two visitor-center pins are excluded because their titles do not identify encounters; the exclusions are listed in the notebook and report.
- Date repairs/inference retain original text and explicit QC flags. One 2026-map record has a 2025 title date and remains flagged rather than silently reassigned.
- The source is live: rerunning the notebook can change row counts, retrieval timestamps, checksums, maps, and report metrics.
- Confirm permission and redistribution terms before promoting narratives or individual-whale details into a published production dataset.
